# Session 4: Simple LSTM API Deployment

This notebook deploys the trained LSTM from Session 2.

**Input:** latest 10 network observations  
**Model:** saved Session 2 LSTM  
**Output:** predicted next RSRP

Required files: `raw_dataset.csv` and `lstm_rsrp_model.keras`.

## 1. Install the libraries

In [3]:
%pip install -q tensorflow pandas numpy scikit-learn fastapi uvicorn httpx folium

Note: you may need to restart the kernel to use updated packages.


## 2. Import the libraries

In [5]:
from pathlib import Path
import time
import threading

import numpy as np
import pandas as pd
import tensorflow as tf
import uvicorn
import folium  # Geospatial mapping library

from sklearn.preprocessing import StandardScaler
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from typing import List

LOOKBACK = 10
MODEL_VERSION = 'LSTM-RSRP-2.0-LOCATION'

C:\Users\asmidar\AppData\Local\anaconda3\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 3. Check the required files
The model was created in Session 2. Session 4 loads the saved model; it does not train it again.

In [7]:
DATA_FILE = Path('raw_dataset.csv')
MODEL_FILE = Path('lstm_rsrp_model.keras')

for file in [DATA_FILE, MODEL_FILE]:
    if not file.exists():
        raise FileNotFoundError(f'Place {file.name} in the same folder as this notebook.')

print('Dataset and trained LSTM model are available.')

Dataset and trained LSTM model are available.


## 4. Recreate the Session 2 preprocessing
The same feature order and scalers must be used during training and deployment.

In [9]:
df = pd.read_csv(DATA_FILE)
required = ['Timestamp', 'SessionID', 'Level', 'Latitude', 'Longitude']
missing = [column for column in required if column not in df.columns]
if missing:
    raise ValueError(f'Missing dataset columns: {missing}')

df['Timestamp'] = pd.to_datetime(
    df['Timestamp'], format='%Y.%m.%d_%H.%M.%S', errors='coerce'
)
feature_candidates = [
    'Level', 'SNR', 'CQI', 'DL_bitrate',
    'UL_bitrate', 'Speed', 'Latitude', 'Longitude'
]
features = [column for column in feature_candidates if column in df.columns]
TARGET = 'Level'

df = df.dropna(subset=required).copy()
df = df.sort_values(['SessionID', 'Timestamp']).reset_index(drop=True)
df[features] = df.groupby('SessionID')[features].transform(lambda values: values.ffill().bfill())
df[features] = df[features].fillna(df[features].median(numeric_only=True))

sessions = df.groupby('SessionID')['Timestamp'].min().sort_values().index.tolist()
training_end = max(1, int(len(sessions) * 0.70))
training_sessions = sessions[:training_end]
training_mask = df['SessionID'].isin(training_sessions)

feature_scaler = StandardScaler().fit(df.loc[training_mask, features])
target_scaler = StandardScaler().fit(df.loc[training_mask, [TARGET]])

print('Features used by the LSTM:', features)
print('Number of observations required:', LOOKBACK)

Features used by the LSTM: ['Level', 'SNR', 'CQI', 'DL_bitrate', 'UL_bitrate', 'Speed', 'Latitude', 'Longitude']
Number of observations required: 10


## 5. Load the trained LSTM

In [11]:
model = tf.keras.models.load_model(MODEL_FILE)
print('Saved LSTM loaded successfully.')
print('Model input shape:', model.input_shape)

expected_shape = (LOOKBACK, len(features))
if tuple(model.input_shape[1:]) != expected_shape:
    raise ValueError(
        f'Model expects {model.input_shape[1:]}, but the API prepared {expected_shape}.'
    )

Saved LSTM loaded successfully.
Model input shape: (None, 10, 8)


## 6. Define the prediction function
This function receives 10 observations, applies the Session 2 scaling and asks the saved LSTM to predict the next RSRP.

In [13]:
def predict_next_rsrp(observations):
    if len(observations) != LOOKBACK:
        raise ValueError(f'Exactly {LOOKBACK} observations are required.')

    input_frame = pd.DataFrame(observations)
    missing_features = [column for column in features if column not in input_frame.columns]
    if missing_features:
        raise ValueError(f'Missing features: {missing_features}')

    scaled = feature_scaler.transform(input_frame[features])
    sequence = scaled.reshape(1, LOOKBACK, len(features)).astype(np.float32)
    predicted_scaled = model.predict(sequence, verbose=0)
    predicted_rsrp = target_scaler.inverse_transform(predicted_scaled)[0, 0]
    return float(predicted_rsrp)

def signal_class(rsrp):
    if rsrp >= -90:
        return 'Good'
    if rsrp >= -105:
        return 'Weak'
    return 'Poor'

## 7. Define the API input
Each observation contains exactly the same eight variables used in the updated Session 2. Latitude and longitude are model input features and also identify the location of the predicted next RSRP.

In [15]:
class Observation(BaseModel):
    Level: float
    SNR: float
    CQI: float
    DL_bitrate: float = Field(ge=0)
    UL_bitrate: float = Field(ge=0)
    Speed: float = Field(ge=0)
    Latitude: float = Field(ge=-90, le=90)
    Longitude: float = Field(ge=-180, le=180)

class PredictionRequest(BaseModel):
    observations: List[Observation]

## 8. Create the FastAPI service
`/sample-request` prepares a complete input automatically, so students do not need to type 10 observations.

In [17]:
app = FastAPI(
    title='Simple LSTM-RSRP API',
    description='Uses 10 network observations to predict the next RSRP.',
    version=MODEL_VERSION
)

@app.get('/health')
def health():
    return {'status': 'ok', 'lstm_loaded': model is not None}

@app.get('/version')
def version():
    return {
        'model_version': MODEL_VERSION,
        'lookback': LOOKBACK,
        'features': features
    }

@app.get('/sample-request')
def sample_request():
    latest_session = df['SessionID'].iloc[-1]
    sample = (
        df[df['SessionID'] == latest_session]
        .sort_values('Timestamp')
        .tail(LOOKBACK)
    )
    if len(sample) < LOOKBACK:
        raise HTTPException(status_code=500, detail='Sample session has fewer than 10 rows.')
    return {'observations': sample[features].to_dict(orient='records')}

@app.post('/predict-rsrp')
def predict_rsrp(request: PredictionRequest):
    start = time.perf_counter()
    try:
        records = [observation.model_dump() for observation in request.observations]
        prediction = predict_next_rsrp(records)
    except ValueError as error:
        raise HTTPException(status_code=400, detail=str(error)) from error

    last_observation = records[-1]
    return {
        'predicted_rsrp_dbm': round(prediction, 2),
        'signal_class': signal_class(prediction),
        'latitude': last_observation['Latitude'],
        'longitude': last_observation['Longitude'],
        'model_version': MODEL_VERSION,
        'response_time_ms': round((time.perf_counter() - start) * 1000, 2)
    }

## 9. Test the API inside Jupyter
This confirms that the API works before it is opened in the browser.

In [19]:
client = TestClient(app)

print('Health:', client.get('/health').json())
sample = client.get('/sample-request').json()
response = client.post('/predict-rsrp', json=sample)
response.raise_for_status()
prediction_result = response.json()
print('Prediction:', prediction_result)

Health: {'status': 'ok', 'lstm_loaded': True}
Prediction: {'predicted_rsrp_dbm': -96.61, 'signal_class': 'Weak', 'latitude': 3.07066924, 'longitude': 101.6052545, 'model_version': 'LSTM-RSRP-2.0-LOCATION', 'response_time_ms': 325.54}


## 10. Display the prediction on a GeoAI map
The LSTM predicts the next RSRP. Folium places the prediction at its latitude and longitude. Green represents good signal, orange represents weak signal and red represents poor signal.

In [21]:
marker_colours = {
    'Good': 'green',
    'Weak': 'orange',
    'Poor': 'red'
}

latitude = prediction_result['latitude']
longitude = prediction_result['longitude']
predicted_rsrp = prediction_result['predicted_rsrp_dbm']
signal = prediction_result['signal_class']

prediction_map = folium.Map(
    location=[latitude, longitude],
    zoom_start=15,
    tiles='OpenStreetMap'
)

popup_text = (
    f'<b>LSTM-RSRP Prediction</b><br>'
    f'Latitude: {latitude:.6f}<br>'
    f'Longitude: {longitude:.6f}<br>'
    f'Predicted RSRP: {predicted_rsrp:.2f} dBm<br>'
    f'Signal class: {signal}<br>'
    f'Model: {prediction_result["model_version"]}'
)

folium.CircleMarker(
    location=[latitude, longitude],
    radius=10,
    color=marker_colours[signal],
    fill=True,
    fill_color=marker_colours[signal],
    fill_opacity=0.80,
    popup=folium.Popup(popup_text, max_width=300),
    tooltip=f'{predicted_rsrp:.2f} dBm - {signal}'
).add_to(prediction_map)

folium.LayerControl().add_to(prediction_map)
prediction_map.save('session4_lstm_prediction_map.html')
print('Map saved as session4_lstm_prediction_map.html')
prediction_map

Map saved as session4_lstm_prediction_map.html


## 11. Run the API
Restart the Jupyter kernel first if port 8000 is already being used by an older API.

In [23]:
def run_api():
    uvicorn.run(app, host='127.0.0.1', port=8001, log_level='info')

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print('API is running.')
print('Open: http://127.0.0.1:8001/docs')

API is running.
Open: http://127.0.0.1:8001/docs


INFO:     Started server process [18164]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8001 (Press CTRL+C to quit)


## Classroom demonstration
1. Open `http://127.0.0.1:8001/docs`.
2. Run `GET /health` to show that the saved LSTM is active.
3. Run `GET /sample-request` and copy the complete response.
4. Paste the response into `POST /predict-rsrp`.
5. Explain the returned next RSRP and signal class.
6. Return to Jupyter and show the prediction on the GeoAI map.

**Deployment flow:** application sends 10 observations → API calls saved LSTM → API returns predicted next RSRP.

## 12. Student deployment dashboard
The Streamlit dashboard lets students visualise the complete deployment process without manually copying JSON in Swagger UI.

It displays API status, the 10 input observations, the request payload, predicted RSRP, signal class, response time, map location and input-versus-prediction chart.

In [26]:
%pip install -q streamlit requests

Note: you may need to restart the kernel to use updated packages.


### Run the dashboard
Keep this notebook and its API running. Open a terminal in the same folder and enter:

```bash
streamlit run student_lstm_deployment_dashboard.py
```

The dashboard normally opens at `http://localhost:8501` and connects to the API at `http://127.0.0.1:8001`.

### Classroom demonstration flow
1. Show that the API and saved LSTM are online.
2. Explain that each table row is one time-ordered network observation.
3. Show the JSON request containing exactly 10 observations.
4. Select **Run prediction**.
5. Explain the predicted next RSRP, signal class and response time.
6. Use the map to connect the model output with its geographic location.
7. Compare the latest input RSRP with the predicted next RSRP.